# 02 — Build compact gold tables
Attach the destination Lakehouse and make the mirrored source tables visible. Writes are deterministic full replacements suitable for the compact demo.

In [ ]:
from pyspark.sql import functions as F

def source(name):
    matches = {r.tableName.lower(): r.tableName for r in spark.sql('SHOW TABLES').collect()}
    if name not in matches:
        raise ValueError(f'Missing source table: {name}')
    df = spark.table(matches[name])
    return df.select(*[F.col(c).alias(c.lower()) for c in df.columns])

tables = {n: source(n) for n in [
    'aircraft','route','flight','booking','customer','loyalty_transaction',
    'cargo_shipment','maintenance_event','crew_roster','disruption_event'
]}

In [ ]:
gold = {
  'gold_dim_aircraft': tables['aircraft'],
  'gold_dim_route': tables['route'],
  'gold_dim_customer': tables['customer'],
  'gold_fact_flight': tables['flight'].withColumn('flight_key', F.concat_ws('|', 'flight_no', F.col('date').cast('string'))),
  'gold_fact_booking': tables['booking'].withColumn('flight_key', F.concat_ws('|', 'flight_no', F.col('date').cast('string'))).withColumn('total_revenue_gbp', F.coalesce('revenue_gbp', F.lit(0)) + F.coalesce('ancillary_gbp', F.lit(0))),
  'gold_fact_disruption': tables['disruption_event'].withColumn('flight_key', F.concat_ws('|', 'flight_no', F.col('date').cast('string'))).withColumn('disruption_cost_gbp', F.coalesce('rebook_cost_gbp', F.lit(0)) + F.coalesce('compensation_gbp', F.lit(0))),
  'gold_fact_cargo': tables['cargo_shipment'].withColumn('flight_key', F.concat_ws('|', 'flight_no', F.col('date').cast('string'))),
  'gold_fact_maintenance': tables['maintenance_event'],
  'gold_fact_loyalty': tables['loyalty_transaction'],
  'gold_fact_crew': tables['crew_roster']
}

for name, df in gold.items():
    (df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(name))
    print(f'{name}: {df.count():,} rows')

In [ ]:
checks = []
for name in gold:
    detail = spark.sql(f'DESCRIBE DETAIL {name}').select('format', 'numFiles', 'sizeInBytes').first()
    checks.append((name, spark.table(name).count(), detail['format'], detail['numFiles'], detail['sizeInBytes']))
display(spark.createDataFrame(checks, ['table', 'rows', 'format', 'files', 'bytes']))
print('Gold build complete. Refresh Direct Lake model metadata if schema changed.')